In [16]:
import sys
import os

# Set the main path in the root folder of the project.
sys.path.append(os.path.join('..'))

In [17]:
# Settings for autoreloading.
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
from src.utils.seed import set_random_seed

# Set the random seed for deterministic operations.
SEED = 42
set_random_seed(SEED)

In [19]:
import torch

# Set the device for training and querying the model.
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'The selected device is: "{DEVICE}"')

The selected device is: "cpu"


In [20]:
import os

BASE_DATA_DIR = os.path.join('..', 'data', 'metr-la')

In [21]:
import pickle
with open(os.path.join(BASE_DATA_DIR, 'processed', 'scaler.pkl'), 'rb') as f:
    scaler = pickle.load(f)

In [22]:
from src.spatial_temporal_gnn.model import SpatialTemporalGNN
from src.data.data_extraction import get_adjacency_matrix

# Get the adjacency matrix
adj_matrix_structure = get_adjacency_matrix(
    os.path.join(BASE_DATA_DIR, 'raw', 'adj_mx.pkl'))

# Get the header of the adjacency matrix, the node indices and the
# matrix itself.
header, node_ids_dict, adj_matrix = adj_matrix_structure

# Get the STGNN and load the checkpoints.
spatial_temporal_gnn = SpatialTemporalGNN(9, 1, 12, 12, adj_matrix, DEVICE, 64)

stgnn_checkpoints_path = os.path.join('..', 'models', 'checkpoints',
                                      'st_gnn_metr_la.pth')

stgnn_checkpoints = torch.load(stgnn_checkpoints_path)
spatial_temporal_gnn.load_state_dict(stgnn_checkpoints['model_state_dict'])

# Set the STGNN in evaluation mode.
spatial_temporal_gnn.eval();

In [23]:
from src.data.data_extraction import get_locations_dataframe

# Get the dataframe containing the latitude and longitude of each sensor.
locations_df = get_locations_dataframe(
    os.path.join(BASE_DATA_DIR, 'raw', 'graph_sensor_locations.csv'),
    has_header=True)

In [24]:
# Get the node positions dictionary.
node_pos_dict = { i: id for id, i in node_ids_dict.items() }

In [25]:
import pickle

# Get the data scaler.
with open(os.path.join(BASE_DATA_DIR, 'processed', 'scaler.pkl'), 'rb') as f:
    scaler = pickle.load(f)

In [26]:
import os
import numpy as np

# Get the data and the values predicted by the STGNN.
x_train = np.load(os.path.join(BASE_DATA_DIR, 'explainable', 'x_train.npy'))
y_train = np.load(os.path.join(BASE_DATA_DIR, 'explainable', 'y_train.npy'))
x_train_time = np.load(os.path.join(BASE_DATA_DIR, 'explainable', 'x_train_time.npy'))
y_train_time = np.load(os.path.join(BASE_DATA_DIR, 'explainable', 'y_train_time.npy'))

x_val = np.load(os.path.join(BASE_DATA_DIR, 'explainable', 'x_val.npy'))
y_val = np.load(os.path.join(BASE_DATA_DIR, 'explainable', 'y_val.npy'))
x_val_time = np.load(os.path.join(BASE_DATA_DIR, 'explainable', 'x_val_time.npy'))
y_val_time = np.load(os.path.join(BASE_DATA_DIR, 'explainable', 'y_val_time.npy'))

x_test = np.load(os.path.join(BASE_DATA_DIR, 'explainable', 'x_test.npy'))
y_test = np.load(os.path.join(BASE_DATA_DIR, 'explainable', 'y_test.npy'))
x_test_time = np.load(os.path.join(BASE_DATA_DIR, 'explainable', 'x_test_time.npy'))
y_test_time = np.load(os.path.join(BASE_DATA_DIR, 'explainable', 'y_test_time.npy'))

FileNotFoundError: [Errno 2] No such file or directory: '..\\data\\metr-la\\explainable\\x_train.npy'

In [ ]:
from src.data.data_processing import get_distance_matrix

if not os.path.exists(
    os.path.join(BASE_DATA_DIR, 'processed', 'distance_matrix.npy')):
    # Build the distance matrix between the nodes.
    distance_matrix = get_distance_matrix(
        adj_matrix,
        locations_df,
        node_pos_dict)

    # Save the distance matrix.
    np.save(os.path.join(BASE_DATA_DIR, 'processed', 'distance_matrix.npy'),
            distance_matrix)

else:
    # Load the distance matrix.
    distance_matrix = np.load(
        os.path.join(BASE_DATA_DIR, 'processed', 'distance_matrix.npy'))

In [ ]:
from src.explanation.monte_carlo.explanation import apply_grid_search

apply_grid_search(
    x_train[::10],
    y_train[::10],
    distance_matrix,
    spatial_temporal_gnn,
    scaler,
    n_rollouts_list=[30, 50],
    explanation_size_factor_list=[2, 3],
    cut_size_factor_list=[2],
    exploration_weight_list=[5, 10, 20],
    remove_value_list=[0.])

Testing: cut_size_factor: 2 explanation_size_factor: 2 exploration_weight: 5 n_rollouts: 30 remove_value: 0.0
[100/100] - 559s - MAE: { severe_congestion 3.01 -congestion 1.35 -free_flow 0.814 - total: 1.69 } - RMSE: { severe_congestion 3.62 -congestion 1.69 -free_flow 1.02 - total: 2.07 } - MAPE: { severe_congestion 13.4% -congestion 2.77% -free_flow 1.25% - total: 5.77% } - Average time: 5.59s 

Testing: cut_size_factor: 2 explanation_size_factor: 2 exploration_weight: 5 n_rollouts: 50 remove_value: 0.0
[100/100] - 940s - MAE: { severe_congestion 3.12 -congestion 1.38 -free_flow 0.753 - total: 1.72 } - RMSE: { severe_congestion 3.78 -congestion 1.72 -free_flow 0.959 - total: 2.11 } - MAPE: { severe_congestion 14% -congestion 2.9% -free_flow 1.15% - total: 5.99% } - Average time: 9.4s 

Testing: cut_size_factor: 2 explanation_size_factor: 2 exploration_weight: 10 n_rollouts: 30 remove_value: 0.0
[100/100] - 576s - MAE: { severe_congestion 3.49 -congestion 1.46 -free_flow 0.828 - total

The best hyperparameters are expressed below.

In [ ]:
CUT_SIZE_FACTOR = 2
EXPLANATION_SIZE_FACTOR = 2
EXPLORATION_WEIGHT = 20
N_ROLLOUTS = 50

In [ ]:
import os
import numpy as np

EXPLAINED_DATA_DIR = os.path.join(BASE_DATA_DIR, 'explained')
os.makedirs(EXPLAINED_DATA_DIR, exist_ok=True)


In [ ]:
from src.explanation.monte_carlo.explanation import get_all_explanations


print('Computing the explanations for the training set...')
x_train_explained, y_train_explained, train_scores = get_all_explanations(
    x_train,
    y_train,
    distance_matrix,
    spatial_temporal_gnn,
    scaler,
    n_rollouts=N_ROLLOUTS,
    explanation_size_factor=EXPLANATION_SIZE_FACTOR,
    cut_size_factor=CUT_SIZE_FACTOR,
    exploration_weight=EXPLORATION_WEIGHT,
    remove_value=0.,
    divide_by_traffic_cluster_kind=True)

# Save the explained data.
np.save(os.path.join(EXPLAINED_DATA_DIR, 'x_train.npy'), x_train_explained)
np.save(os.path.join(EXPLAINED_DATA_DIR, 'y_train.npy'), y_train_explained)
np.save(os.path.join(EXPLAINED_DATA_DIR, 'train_scores.npy'), train_scores)
np.save(os.path.join(EXPLAINED_DATA_DIR, 'x_train_time.npy'), x_train_time)
np.save(os.path.join(EXPLAINED_DATA_DIR, 'y_train_time.npy'), y_train_time)


Computing the explanations for the training set...
[999/999] - 10845s - MAE: { severe_congestion 3.32 -congestion 1.5 -free_flow 0.668 - total: 1.83 } - RMSE: { severe_congestion 4.07 -congestion 1.87 -free_flow 0.835 - total: 2.25 } - MAPE: { severe_congestion 14.9% -congestion 3.07% -free_flow 1.02% - total: 6.34% } - Average time: 10.9s 


In [ ]:
from src.explanation.monte_carlo.explanation import get_all_explanations


print('Computing the explanations for the validation set...')
x_val_explained, y_val_explained, val_scores = get_all_explanations(
    x_val,
    y_val,
    distance_matrix,
    spatial_temporal_gnn,
    scaler,
    n_rollouts=N_ROLLOUTS,
    explanation_size_factor=EXPLANATION_SIZE_FACTOR,
    cut_size_factor=CUT_SIZE_FACTOR,
    exploration_weight=EXPLORATION_WEIGHT,
    remove_value=0.,
    divide_by_traffic_cluster_kind=True)

# Save the explained data.
np.save(os.path.join(EXPLAINED_DATA_DIR, 'x_val.npy'), x_val_explained)
np.save(os.path.join(EXPLAINED_DATA_DIR, 'y_val.npy'), y_val_explained)
np.save(os.path.join(EXPLAINED_DATA_DIR, 'val_scores.npy'), val_scores)
np.save(os.path.join(EXPLAINED_DATA_DIR, 'x_val_time.npy'), x_val_time)
np.save(os.path.join(EXPLAINED_DATA_DIR, 'y_val_time.npy'), y_val_time)

Computing the explanations for the validation set...
[198/198] - 1871s - MAE: { severe_congestion 3.63 -congestion 1.39 -free_flow 0.668 - total: 1.86 } - RMSE: { severe_congestion 4.32 -congestion 1.73 -free_flow 0.848 - total: 2.26 } - MAPE: { severe_congestion 17.2% -congestion 2.75% -free_flow 1.02% - total: 6.88% } - Average time: 9.45s 


In [ ]:
from src.explanation.monte_carlo.explanation import get_all_explanations


print('Computing the explanations for the test set...')
x_test_explained, y_test_explained, test_scores = get_all_explanations(
    x_test,
    y_test,
    distance_matrix,
    spatial_temporal_gnn,
    scaler,
    n_rollouts=N_ROLLOUTS,
    explanation_size_factor=EXPLANATION_SIZE_FACTOR,
    cut_size_factor=CUT_SIZE_FACTOR,
    exploration_weight=EXPLORATION_WEIGHT,
    remove_value=0.,
    divide_by_traffic_cluster_kind=True)

# Save the explained data.
np.save(os.path.join(EXPLAINED_DATA_DIR, 'x_test.npy'), x_test_explained)
np.save(os.path.join(EXPLAINED_DATA_DIR, 'y_test.npy'), y_test_explained)
np.save(os.path.join(EXPLAINED_DATA_DIR, 'test_scores.npy'), test_scores)
np.save(os.path.join(EXPLAINED_DATA_DIR, 'x_test_time.npy'), x_test_time)
np.save(os.path.join(EXPLAINED_DATA_DIR, 'y_test_time.npy'), y_test_time)

Computing the explanations for the test set...
[300/300] - 2663s - MAE: { severe_congestion 3.21 -congestion 1.65 -free_flow 0.713 - total: 1.84 } - RMSE: { severe_congestion 3.82 -congestion 2.06 -free_flow 0.881 - total: 2.24 } - MAPE: { severe_congestion 14.1% -congestion 3.45% -free_flow 1.09% - total: 6.14% } - Average time: 8.88s 


In [ ]:
import os
import numpy as np

# Get the data and the values predicted by the STGNN.
x_train_explained = np.load(os.path.join(BASE_DATA_DIR, 'explained', 'x_train.npy'))

x_val_explained = np.load(os.path.join(BASE_DATA_DIR, 'explained', 'x_val.npy'))

x_test_explained = np.load(os.path.join(BASE_DATA_DIR, 'explained', 'x_test.npy'))

In [ ]:
from src.explanation.monte_carlo.explanation import print_all_fidelity_plus


print('Computing the fidelity+ for the training set...')
print_all_fidelity_plus(x_train, y_train, x_train_explained, spatial_temporal_gnn, scaler, remove_value=0.)
print()

print('Computing the fidelity+ for the validation set...')
print_all_fidelity_plus(x_val, y_val, x_val_explained, spatial_temporal_gnn, scaler, remove_value=0.)
print()

print('Computing the fidelity+ for the test set...')
print_all_fidelity_plus(x_test, y_test, x_test_explained, spatial_temporal_gnn, scaler, remove_value=0.)
print()

Computing the fidelity+ for the training set...
MAE+: { severe_congestion 18.7 -congestion 9.52 -free_flow 2.46 - total: 10.2% } -RMSE+: { severe_congestion 20.5 -congestion 10.4 -free_flow 2.94 - total: 11.3% } -MAPE+: { severe_congestion 83.7% -congestion 19.2% -free_flow 3.79% - total: 35.6% }

Computing the fidelity+ for the validation set...
MAE+: { severe_congestion 17.6 -congestion 8.36 -free_flow 2.25 - total: 9.41% } -RMSE+: { severe_congestion 19.7 -congestion 9.15 -free_flow 2.69 - total: 10.5% } -MAPE+: { severe_congestion 85.7% -congestion 16.4% -free_flow 3.45% - total: 35.2% }

Computing the fidelity+ for the test set...
MAE+: { severe_congestion 19.1 -congestion 9.68 -free_flow 2.96 - total: 10.6% } -RMSE+: { severe_congestion 20.9 -congestion 10.6 -free_flow 3.5 - total: 11.7% } -MAPE+: { severe_congestion 86.2% -congestion 20.4% -free_flow 4.54% - total: 37.1% }



In [ ]:
from src.explanation.monte_carlo.explanation import print_all_sparsity


print('Computing the sparsity for the training set...')
print_all_sparsity(x_train, x_train_explained)
print()

print('Computing the sparsity for the validation set...')
print_all_sparsity(x_val, x_val_explained)
print()

print('Computing the sparsity for the test set...')
print_all_sparsity(x_test, x_test_explained)
print()

Computing the sparsity for the training set...
Sparsity: { severe_congestion 0.984 -congestion 0.988 -free_flow 0.985 - total: 0.986 }

Computing the sparsity for the validation set...
Sparsity: { severe_congestion 0.985 -congestion 0.988 -free_flow 0.985 - total: 0.986 }

Computing the sparsity for the test set...
Sparsity: { severe_congestion 0.983 -congestion 0.987 -free_flow 0.984 - total: 0.985 }

